# Gao et al. (2018): the earlier attempt's findings, checked on DANDI:000007

Attempt `gao-2018-02`, made blind, is compared with `gao-2018-01` in the reconciliation block of
`attempts/gao-2018-02.yaml` (docs/protocol.md, "Repeat attempts"). Findings 1, 3, 4, 6, 11 and 12 of
`gao-2018-01` are reported again by `gao-2018-02` and are not checked here. This notebook checks the
other six (2, 5, 7, 8, 9 and 10) directly, on the dataset and version the earlier attempt used:
[DANDI:000007](https://dandiarchive.org/dandiset/000007/0.220126.1903), version 0.220126.1903, 54 NWB
files streamed with remfile and h5py. It reads no other dataset. Each section opens with the earlier
statement and ends with a short result.

Paper: Gao Z, Davis C, Thomas AM, Economo MN, Abrego AM, Svoboda K, De Zeeuw CI, Li N (2018). A
cortico-cerebellar loop for motor planning. *Nature* 563, 113-116, doi:10.1038/s41586-018-0633-x.

In [ ]:
import json
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import h5py
import numpy as np
import pandas as pd
import remfile
import requests
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

DANDISET_ID, DANDISET_VERSION = "000007", "0.220126.1903"


def _s(x):
    return x.decode() if isinstance(x, bytes) else str(x)


def load_one(asset):
    """One NWB file: trials, spike times, observation intervals, waveforms and the metadata checked below."""
    url = requests.head(f"https://api.dandiarchive.org/api/assets/{asset['asset_id']}/download/",
                        allow_redirects=True, timeout=60).url
    with h5py.File(remfile.File(url), "r") as f:
        tr = f["intervals/trials"]
        trials = pd.DataFrame({k: tr[k][:] for k in ["start_time", "stop_time", "pole_in_time", "pole_out_time",
                                                     "cue_time", "lick_early", "response", "photo_stim_id"]})
        for k in ("response", "photo_stim_id"):
            trials[k] = trials[k].map(_s)
        u = f["units"]
        st, idx = u["spike_times"][:], u["spike_times_index"][:]
        oi, oidx = u["obs_intervals"][:], u["obs_intervals_index"][:]
        sub = f["general/subject"]
        out = dict(
            path=asset["path"], subject=_s(sub["subject_id"][()]),
            subject_fields={k: _s(sub[k][()]) for k in sub},
            session_description=_s(f["session_description"][()]),
            trials=trials, trial_descr={k: _s(tr[k].attrs.get("description", "")) for k in tr},
            spikes=np.split(st, idx[:-1]), obs=np.split(oi, oidx[:-1]),
            cell_type=u["cell_type"][:].astype(str), wf_rate=u["sampling_rate"][:],
            waveform=u["waveform_mean"][:], devices=list(f["general/devices"]),
            egroups={k: json.loads(f[f"general/extracellular_ephys/{k}"].attrs.get("location", "{}"))
                     for k in f["general/extracellular_ephys"] if k != "electrodes"},
            ogen={k: json.loads(_s(f[f"general/optogenetics/{k}/location"][()]) or "{}")
                  for k in f["general/optogenetics"]},
            acquisition=list(f["acquisition"]),
            stimulus=list(f["stimulus/presentation"]) + list(f["stimulus/templates"]),
            processing=list(f["processing"]))
    out["name"] = asset["path"].split("/")[1][4:-4]
    return out


started = time.time()
r = requests.get(f"https://api.dandiarchive.org/api/dandisets/{DANDISET_ID}/versions/{DANDISET_VERSION}"
                 "/assets/?page_size=200", timeout=60).json()
assets = sorted([a for a in r["results"] if a["path"].endswith(".nwb")], key=lambda a: a["path"])
with ThreadPoolExecutor(8) as ex:
    S = list(ex.map(load_one, assets))
for s in S:
    codes = set(s["trials"].photo_stim_id)
    s["group"] = "fastigial" if "1" in codes else "dentate" if "3" in codes else "inhibition"
print(f"DANDI:{DANDISET_ID} version {DANDISET_VERSION}: {len(S)} NWB files streamed in {time.time() - started:.0f} s")

## Data check

In [ ]:
D = pd.DataFrame([dict(subject=s["subject"], group=s["group"], site=",".join(s["ogen"]), trials=len(s["trials"]),
                       units=len(s["spikes"]), pyramidal=int((s["cell_type"] == "pyramidal").sum()),
                       codes=",".join(sorted(set(s["trials"].photo_stim_id)))) for s in S])
print(D.groupby(["group", "subject"]).agg(sessions=("trials", "size"), trials=("trials", "sum"), units=("units", "sum"),
                                          pyramidal=("pyramidal", "sum"), site=("site", "first"),
                                          codes=("codes", "first")).to_string())
print("total:", len(S), "sessions,", D.subject.nunique(), "mice,", D.units.sum(), "units,", D.trials.sum(), "trials")
T = pd.concat([s["trials"] for s in S])
print("pole_in_time - start_time (s):", (T.pole_in_time - T.start_time).describe().round(3).to_dict())
print("responses:", T.response.value_counts().to_dict())

## Earlier finding 2: the photostimulation codes

> photo_stim_id has no description, and the files hold no laser trace. The codes are 1 and 2 in fastigial, 3 and 4 in dentate and 5 and 6 in L7-cre x Ai32 sessions; from the ALM response, 1 and 3 start at sample onset and 2, 4, 5 and 6 at delay onset. Codes 5 and 6 both last through the delay; the ALM change to code 5 fades over the delay (mean absolute change 3.1 to 3.2 spikes/s early in the delay, 2.1 at its end) and to code 6 does not (2.95 to 3.00 early, 2.93 at its end), as two powers would give, but which is 1.5 mW and which 4.5 mW cannot be told, nor the photo-activation power. The ALM change to CN photo-inhibition starts about 30 ms after pole_out_time under both codes, so when the photostimulus starts relative to pole_out_time cannot be checked.

In [ ]:
print("description of photo_stim_id in the 54 files:", pd.Series([s["trial_descr"]["photo_stim_id"] for s in S]).value_counts().to_dict())
print("files with anything under acquisition, stimulus or processing:",
      sum(bool(s["acquisition"] or s["stimulus"] or s["processing"]) for s in S))
print("devices:", pd.Series([", ".join(s["devices"]) for s in S]).value_counts().to_dict())
print("codes by group:", D.groupby("group").codes.agg(lambda x: sorted(set(x))).to_dict())

**Result: still present.** In all 54 files `photo_stim_id` has the description "varchar(8) #" and
nothing else, and no file holds anything under acquisition, stimulus or processing, so there is no laser
trace. The codes are as the earlier finding gives them (1 and 2, 3 and 4, 5 and 6 by group), and nothing in
the files says what each means.

## Earlier finding 5: subject identifiers and descriptions

> Subjects anm00314746, anm00314756, anm00314757, anm00314758 and anm00314760 are named AN105 to AN109 in the paper; by site, anm00314756 is AN106 and anm00314757 is AN107, and the other three cannot be told apart. The genotype of the wild-type mice is "unknown", and session_description is empty.

In [ ]:
subj = pd.DataFrame([dict(subject=s["subject"], fields=", ".join(sorted(s["subject_fields"])),
                          genotype=s["subject_fields"].get("genotype"), site=",".join(
                              f"{k} {v.get('hemisphere')}" for k, v in s["ogen"].items()),
                          alm_ml=",".join(g.get("rec_coordinate_ml", "") for g in s["egroups"].values()))
                     for s in S]).drop_duplicates()
print(subj.to_string())
print("files with an empty session_description:", sum(s["session_description"] == "" for s in S), "of", len(S))
print("any subject field naming AN105 to AN109:",
      any("AN10" in v for s in S for v in s["subject_fields"].values()))

**Result: still present.** The five anm subjects carry only their anm identifiers; no subject field names
AN105 to AN109. Their sites (printed) are as the earlier finding says: anm00314756 alone is dentate and
anm00314757 alone has a left fastigial site and right ALM, while anm00314746, anm00314758 and
anm00314760 share a right fastigial site and left ALM, so they cannot be told apart. The genotype is
"unknown" for every mouse that is not L7-cre x Ai32, and session_description is empty in all 54 files.

## Earlier finding 7: cell types against spike width, and the waveform sampling rate

> The cell_type column labels 60 units "pyramidal" whose mean waveform has a trough-to-peak width under the paper's 0.45 ms (33 under 0.35 ms), and 3 units "unknown"; the waveforms are sampled at a stated 20 kHz in every file, including the 34 sessions recorded with the 32-channel NeuroNexus probe, which the Methods say were sampled at 25 kHz (the other 20 used the 64-channel probe).

In [ ]:
def trough_to_peak_ms(wf, rate):
    tr = int(np.argmin(wf))
    return (tr + int(np.argmax(wf[tr:])) - tr) / rate * 1000

rows = []
for s in S:
    probe = ", ".join(g["probe_type"] for g in s["egroups"].values())
    for i in range(len(s["cell_type"])):
        rows.append(dict(session=s["name"], probe=probe, rate=int(s["wf_rate"][i]), samples=s["waveform"].shape[1],
                         cell_type=s["cell_type"][i], width=trough_to_peak_ms(s["waveform"][i], s["wf_rate"][i])))
W = pd.DataFrame(rows)
W["by_width"] = np.where(W.width > 0.45, "> 0.45 ms", np.where(W.width < 0.35, "< 0.35 ms", "0.35 to 0.45 ms"))
print(pd.crosstab(W.cell_type, W.by_width, margins=True))
print("width range", W.width.min(), "to", W.width.max(), "ms")
print(W.groupby("probe").agg(sessions=("session", "nunique"), stated_rates=("rate", lambda x: sorted(set(x))),
                             samples_per_waveform=("samples", lambda x: sorted(set(x)))).to_string())
print(W.drop_duplicates("session").groupby(["probe", "samples"]).size())

**Result: still present.** 60 units labelled "pyramidal" have a trough-to-peak width under 0.45 ms (27
between 0.35 and 0.45 ms and 33 under 0.35 ms), and 3 units are labelled "unknown". Every unit's waveform
is stated as sampled at 20000 Hz, in the 34 sessions recorded with the 32-channel NeuroNexus probe as in the
20 with the 64-channel probe. In 14 of the 34 NeuroNexus sessions the mean waveforms have 37 or 38 samples
against 30 in every other session, which a higher sampling rate would give for the same duration; the files
do not say which rate applies.

## Earlier finding 8: sessions whose response is not shown to be locked to the trial events

> In 5 of 54 sessions the response to delay photostimulation could not be shown to be locked to the trial events (BAYLORCD12 20180125 and 20180129, BAYLORCD13 20171211 and 20171215, anm00314756 20151023), mostly sessions with few units or photostimulation trials; the other 49 show it. Leaving them out changes no outcome.

In [ ]:
def counts(sp, a, b):
    return (np.searchsorted(sp, b) - np.searchsorted(sp, a)).astype(float)


def frac_sig(C, a, b, alpha):
    p = stats.ttest_ind(C[:, a], C[:, b], axis=1).pvalue
    return float(np.mean(np.nan_to_num(p, nan=1.0) < alpha)) if len(C) else np.nan


DELAY = {"fastigial": ["2"], "dentate": ["4"], "inhibition": ["5", "6"]}
rng8 = np.random.default_rng(8)
ck = []
for s in S:
    t = s["trials"]
    code_, resp = t.photo_stim_id.to_numpy(), t.response.to_numpy()
    # (a) the earlier attempt's contrast: neurons whose rate differs between control and delay photostimulation
    # trials in the 400 ms after pole_out_time, against the 400 ms before it and against labels shifted by a trial
    ctl, st, on = code_ == "0", np.isin(code_, DELAY[s["group"]]), t.pole_out_time.to_numpy()
    pre = np.array([counts(x, on - 0.4, on) for x in s["spikes"]])
    post = np.array([counts(x, on, on + 0.4) for x in s["spikes"]])
    sh = np.roll(st, 1)
    a = dict(before=frac_sig(pre, ctl, st, 0.01), after=frac_sig(post, ctl, st, 0.01),
             after_shifted=frac_sig(post, ~sh, sh, 0.01))
    # (b) trial-type selectivity: neurons whose rate differs between correct lick-right and lick-left control
    # trials in the 1 s before pole_in_time and the 2.6 s after it, against 200 label permutations
    m = (code_ == "0") & (t.lick_early == 0).to_numpy() & np.isin(resp, ["HitR", "HitL"])
    R, L = m & (resp == "HitR"), m & (resp == "HitL")
    pi = t.pole_in_time.to_numpy()
    pre_b = np.array([counts(x, pi - 1.0, pi) for x in s["spikes"]])
    post_b = np.array([counts(x, pi, pi + 2.6) for x in s["spikes"]])
    idx, perm = np.flatnonzero(m), []
    for _ in range(200):
        lab = rng8.permutation(resp[idx])
        pa, pb = np.zeros(len(t), bool), np.zeros(len(t), bool)
        pa[idx[lab == "HitR"]], pb[idx[lab == "HitL"]] = True, True
        perm.append(frac_sig(post_b, pa, pb, 0.05))
    b = dict(sel_before=frac_sig(pre_b, R, L, 0.05), sel_after=frac_sig(post_b, R, L, 0.05),
             sel_perm95=float(np.quantile(perm, 0.95)))
    ck.append(dict(session=s["name"], group=s["group"], units=len(s["spikes"]), trials=len(t),
                   stim_trials=int(st.sum()), right=int(R.sum()), left=int(L.sum()), **a, **b))
CK = pd.DataFrame(ck)
CK["shown_a"] = (CK.after > CK.before) & (CK.after > CK.after_shifted) & (CK.after > 0)
CK["shown_b"] = (CK.sel_after > CK.sel_before) & (CK.sel_after > CK.sel_perm95)
print(CK.round(3).to_string())
EARLIER = ["BAYLORCD12_ses-20180125T191601", "BAYLORCD12_ses-20180129T160551", "BAYLORCD13_ses-20171211T135437",
           "BAYLORCD13_ses-20171215T133939", "anm00314756_ses-20151023T141029"]
print("shown by (a):", int(CK.shown_a.sum()), "of", len(CK), "; not shown:", CK.loc[~CK.shown_a, "session"].tolist())
print("shown by (b):", int(CK.shown_b.sum()), "of", len(CK), "; not shown:", CK.loc[~CK.shown_b, "session"].tolist())
print("shown by neither:", CK.loc[~CK.shown_a & ~CK.shown_b, "session"].tolist())
print("the earlier attempt's five sessions:")
print(CK.set_index("session").loc[EARLIER, ["units", "stim_trials", "right", "left", "shown_a", "shown_b"]].to_string())

**Result: still present, in two of the five sessions.** With the earlier attempt's contrast (a), 50 of
54 sessions show a locked response; not BAYLORCD13 20171211 and 20171215 or anm00314756 20151020 and
20151023. With trial-type selectivity (b), 48 of 54 do. Neither contrast shows it in BAYLORCD13 20171211,
which holds 15 trials (8 with delay photostimulation, and 1 and 4 correct lick-right and lick-left control
trials), or in BAYLORCD13 20171215, which holds 7 units. Of the earlier five sessions, BAYLORCD12 20180125
and 20180129 are shown here by both contrasts and anm00314756 20151023 by (b).

## Earlier finding 9: BAYLORCD8 20171020 after the end of the recording

> In BAYLORCD8 20171020 the behavior runs on after the recording ends. The units' observation intervals end at 7221.8 s, and the last 127 of 462 trials, from 7223.1 to 8481.4 s, hold no spike of any unit. The obs_intervals column records this correctly, but an analysis that does not apply it counts those trials as silent (this attempt's first run did, which lowered the photo-inhibition sessions' mean rate).

In [ ]:
s = [s for s in S if s["name"].startswith("BAYLORCD8_ses-20171020")][0]
t = s["trials"]
ends = np.array([o[:, 1].max() for o in s["obs"]])
allsp = np.concatenate(s["spikes"])
after = (t.start_time > ends.max()).to_numpy()
inside = sum(((allsp >= a) & (allsp < b)).sum() for a, b in zip(t.start_time[after], t.stop_time[after]))
print(s["name"], ":", len(s["spikes"]), "units,", len(t), "trials from", t.start_time.min(), "to", t.stop_time.max(), "s")
print("end of the units' observation intervals: min", ends.min(), "max", ends.max(), "s; last spike", allsp.max(), "s")
print("trials starting after it:", int(after.sum()), "from", t.start_time[after].min(), "to", t.stop_time[after].max(),
      "s; spikes of any unit in them:", int(inside))

**Result: still present.** Every unit's observation interval ends at 7221.77 s, and the last 127 of the
462 trials, from 7223.1 to 8481.39 s, hold no spike of any unit. The obs_intervals column records the
end of the recording.

## Earlier finding 10: the preference trials and epoch of the population selectivity

> For the population selectivity of Fig. 2c (E3), which epoch fixes each neuron's preferred trial type, and are the 10 preference trials 10 in all or 10 of each type? On these data the epoch decides whether the maximum lies in the response epoch (response epoch) or just before the go cue (sample or delay epoch); the paper says only that the epoch gave "qualitatively similar preparatory activity", which holds here. For Fig. 3f the random draw of the preference trials matters as much as the trial count; in the last 500 ms of the delay, 15 of 20 draws (5 of each type) and 13 of 20 (10 of each type) bring ALM selectivity under CN photo-inhibition to zero.

In [ ]:
GRID = np.arange(-3.0, 1.31, 0.01)       # time from the go cue (cue_time), s
neurons = []
for s in S:
    t = s["trials"]
    resp, code_ = t.response.to_numpy(), t.photo_stim_id.to_numpy()
    instr = np.array([x[-1] if x.startswith(("Hit", "Err")) else "" for x in resp])
    pi, po, cue = t.pole_in_time.to_numpy(), t.pole_out_time.to_numpy(), t.cue_time.to_numpy()
    for j, sp in enumerate(s["spikes"]):
        if s["cell_type"][j] != "pyramidal":
            continue
        o = s["obs"][j]
        m = ((t.lick_early == 0).to_numpy() & (instr != "") & (t.start_time >= o[:, 0].min()).to_numpy()
             & (t.stop_time <= o[:, 1].max()).to_numpy())
        corr = m & (code_ == "0") & np.isin(resp, ["HitR", "HitL"])
        ep = {"sample": counts(sp, pi, po) / (po - pi), "delay": counts(sp, po, cue) / (cue - po),
              "response": counts(sp, cue, cue + 1.3) / 1.3}
        R, L = corr & (resp == "HitR"), corr & (resp == "HitL")
        if R.sum() < 2 or L.sum() < 2:
            continue
        sel = any(np.nan_to_num(stats.ttest_ind(v[R], v[L]).pvalue, nan=1.0) < 0.05 for v in ep.values())
        neurons.append(dict(sp=sp, m=m, corr=corr, resp=resp, instr=instr, code=code_, cue=cue, ep=ep, sel=sel,
                            group=s["group"]))
print(len(neurons), "pyramidal neurons with 2 or more correct control trials of each type;",
      sum(n["sel"] for n in neurons), "selective (t-test, p < 0.05, sample, delay or response epoch)")


def psth(n, mask):
    rel = np.sort(np.concatenate([n["sp"][(n["sp"] > c - 3.2) & (n["sp"] < c + 1.5)] - c for c in n["cue"][mask]]))
    return (np.searchsorted(rel, GRID + 0.1) - np.searchsorted(rel, GRID - 0.1)) / 0.2 / mask.sum()


# Fig. 2c: preferred type from screening trials (10 of each type, or 5 of each, 10 in all) by the count in one
# epoch; selectivity on the remaining correct control trials, 200 ms boxcar; 5 draws each.
res = []
for per_type in (10, 5):
    for epoch in ("sample", "delay", "response"):
        for draw in range(5):
            rng = np.random.default_rng(1000 * per_type + draw)
            rows = []
            for n in neurons:
                if not n["sel"]:
                    continue
                R = np.flatnonzero(n["corr"] & (n["resp"] == "HitR"))
                L = np.flatnonzero(n["corr"] & (n["resp"] == "HitL"))
                if len(R) <= per_type + 5 or len(L) <= per_type + 5:
                    continue
                sr, sl = rng.choice(R, per_type, replace=False), rng.choice(L, per_type, replace=False)
                pref = "R" if n["ep"][epoch][sr].mean() >= n["ep"][epoch][sl].mean() else "L"
                rest = n["corr"].copy()
                rest[sr], rest[sl] = False, False
                rows.append(psth(n, rest & (n["resp"] == "Hit" + pref))
                            - psth(n, rest & (n["resp"] == ("HitL" if pref == "R" else "HitR"))))
            A = np.array(rows).mean(0)
            res.append(dict(per_type=per_type, epoch=epoch, draw=draw, neurons=len(rows),
                            t_max=round(float(GRID[np.argmax(A)]), 2), max=float(A.max())))
F2 = pd.DataFrame(res)
print("Time of the maximum population selectivity, s from the go cue (response epoch 0 to 1.3 s):")
print(F2.groupby(["per_type", "epoch"]).agg(neurons=("neurons", "first"), t_max_min=("t_max", "min"),
                                             t_max_max=("t_max", "max"), draws_in_response=("t_max", lambda x: int((x > 0).sum())),
                                             max_min=("max", "min"), max_max=("max", "max")).round(2).to_string())
print("t_max by draw:")
print(F2.pivot_table(index=["per_type", "epoch"], columns="draw", values="t_max").to_string())

In [ ]:
# Fig. 3f: selective neurons of the photo-inhibition mice; preferred type from screening trials by the delay
# count; last 500 ms of the delay, control against photo-inhibition trials (all trials by instructed type);
# neurons with more than 3 trials in each of the four conditions; bootstrap 95% interval over neurons;
# abolished when the control interval is above zero and the photo-inhibition interval contains zero. 20 draws.
inh = [n for n in neurons if n["group"] == "inhibition" and n["sel"]]
res = []
for codes in (["5", "6"], ["5"], ["6"]):
    for per_type in (5, 10):
        for draw in range(20):
            rng = np.random.default_rng(50000 + 100 * per_type + draw)
            Cs, Ps = [], []
            for n in inh:
                R = np.flatnonzero(n["corr"] & (n["resp"] == "HitR"))
                L = np.flatnonzero(n["corr"] & (n["resp"] == "HitL"))
                if len(R) < per_type or len(L) < per_type:
                    continue
                sr, sl = rng.choice(R, per_type, replace=False), rng.choice(L, per_type, replace=False)
                rate = counts(n["sp"], n["cue"] - 0.5, n["cue"]) / 0.5
                pref = "R" if n["ep"]["delay"][sr].mean() >= n["ep"]["delay"][sl].mean() else "L"
                non = "L" if pref == "R" else "R"
                base = n["m"].copy()
                base[sr], base[sl] = False, False
                ctl, ph = base & (n["code"] == "0"), base & np.isin(n["code"], codes)
                g = [ctl & (n["instr"] == pref), ctl & (n["instr"] == non), ph & (n["instr"] == pref), ph & (n["instr"] == non)]
                if min(x.sum() for x in g) <= 3:
                    continue
                Cs.append(rate[g[0]].mean() - rate[g[1]].mean())
                Ps.append(rate[g[2]].mean() - rate[g[3]].mean())
            Cs, Ps = np.array(Cs), np.array(Ps)
            bi = rng.integers(0, len(Cs), (1000, len(Cs)))
            c_lo = np.quantile(Cs[bi].mean(1), 0.025)
            p_lo, p_hi = np.quantile(Ps[bi].mean(1), [0.025, 0.975])
            res.append(dict(codes="+".join(codes), per_type=per_type, draw=draw, neurons=len(Cs), control=Cs.mean(),
                            control_lo=c_lo, photo=Ps.mean(), photo_lo=p_lo, photo_hi=p_hi,
                            abolished=bool(c_lo > 0 and p_lo <= 0 <= p_hi)))
F3 = pd.DataFrame(res)
print(F3.groupby(["codes", "per_type"]).agg(neurons=("neurons", "first"), draws=("draw", "size"),
                                             abolished=("abolished", "sum"), photo_min=("photo", "min"),
                                             photo_max=("photo", "max"), control_lo_min=("control_lo", "min")).round(3).to_string())
print(f"run time {time.time() - started:.0f} s")

**Result: still present.** Fig. 2c: over 1274 selective neurons, the maximum of the population
selectivity lies in the response epoch (0.24 to 0.49 s after the go cue) in every draw when the response
epoch fixes the preferred type, 0.06 to 0.23 s before the go cue when the delay epoch does, and in the
sample epoch (1.60 to 2.00 s before the go cue) in 9 of 10 draws when the sample epoch does, with 5 or
10 screening trials of each type. Which epoch fixes the preference decides where the maximum lies. Fig. 3f:
with codes 5 and 6 pooled, or code 5 alone, no draw of 20 brings the photo-inhibition selectivity's interval
to include zero. With code 6 alone, 8 of 20 draws with 5 screening trials of each type and 11 of 20 with 10
of each do, and the photo-inhibition selectivity ranges from 0.067 to 0.460 and from 0.055 to 0.356
spikes/s across draws, so there the draw decides the outcome.